# 03 · REINFORCE from scratch

**Teach a Small AI to Think** · unicodesnwrites.tech

You will build the simplest learning-from-a-score algorithm, REINFORCE (Williams, 1992), in a few lines of numpy, and watch a tiny "model" learn that 3 + 4 = 7 without ever being told the answer. Then you will see why subtracting a **baseline** matters so much.

**How to open it in Colab:** go to [colab.research.google.com](https://colab.research.google.com), choose **File → Upload notebook**, and pick this file. No GPU needed: the default CPU runtime is fine.

**Expected runtime:** under a minute.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. The world

One question, four possible answers, and a score sheet the policy never sees:

| answer | score | why |
|---|---|---|
| `7` | 1.5 | correct (1.0) + wrote a number (0.5) |
| `8` | 0.5 | wrote a number |
| `6` | 0.5 | wrote a number |
| `no idea` | 0.0 | nothing |

In [ ]:
ANSWERS = ["7", "8", "6", "no idea"]
REWARDS = np.array([1.5, 0.5, 0.5, 0.0])

def softmax(logits):
    z = logits - logits.max()
    e = np.exp(z)
    return e / e.sum()

print(softmax(np.zeros(4)))  # no preference yet: 25% each

## 2. One REINFORCE step

1. Sample a batch of answers from the current probabilities.
2. Score them.
3. Work out each answer's **advantage**: its score minus the batch average (or the raw score, with no baseline).
4. Nudge the logits: `onehot(k) - probs` points toward "make answer k more likely". Scale it by the advantage and take a small step.

In [ ]:
def reinforce_step(logits, rng, batch_size=8, lr=0.5, baseline="mean", bonus=0.0):
    probs = softmax(logits)
    picks = rng.choice(len(ANSWERS), size=batch_size, p=probs)
    rewards = REWARDS[picks] + bonus
    advantages = rewards - rewards.mean() if baseline == "mean" else rewards
    grad = np.zeros_like(logits)
    for k, a in zip(picks, advantages):
        grad += a * (np.eye(len(ANSWERS))[k] - probs)
    return logits + lr * grad / batch_size, rewards.mean()

rng = np.random.default_rng(0)
logits = np.zeros(4)
for step in range(3):
    logits, avg = reinforce_step(logits, rng)
    print(f"step {step + 1}: probs {np.round(softmax(logits), 3)}  batch mean reward {avg:.2f}")

## 3. Train, and watch the right answer win

We repeat the step 60 times and track the probability of `7`.

In [ ]:
def train(steps=60, batch_size=8, lr=0.5, baseline="mean", bonus=0.0, seed=0):
    rng = np.random.default_rng(seed)
    logits = np.zeros(len(ANSWERS))
    history = [softmax(logits)[0]]
    for _ in range(steps):
        logits, _ = reinforce_step(logits, rng, batch_size, lr, baseline, bonus)
        history.append(softmax(logits)[0])
    return np.array(history)

h = train()
print(f"P(7) at start {h[0]:.2f}, after 20 steps {h[20]:.2f}, after 60 steps {h[60]:.2f}")

## 4. The baseline experiment

Now the interesting part. We add **+10 points to every answer**, as if the grader gave everyone points for showing up. This changes nothing about which answer is best. We run each setup with 20 different random seeds and plot the average (line) and the spread between runs (shaded band).

In [ ]:
def many_runs(baseline, bonus, seeds=20):
    return np.stack([train(baseline=baseline, bonus=bonus, seed=s) for s in range(seeds)])

setups = [("no baseline", "none", 0.0), ("no baseline, +10 for everyone", "none", 10.0),
          ("mean baseline, +10 for everyone", "mean", 10.0)]

fig, ax = plt.subplots(figsize=(8, 4.5))
for label, baseline, bonus in setups:
    runs = many_runs(baseline, bonus)
    mean, std = runs.mean(axis=0), runs.std(axis=0)
    ax.plot(mean, label=label)
    ax.fill_between(range(len(mean)), np.clip(mean - std, 0, 1), np.clip(mean + std, 0, 1), alpha=0.2)
    print(f"{label:<34} P(7) after 60 steps: {mean[-1]:.2f} (spread {std[-1]:.2f})")
ax.set_xlabel("training step")
ax.set_ylabel("probability of answering 7")
ax.set_ylim(0, 1)
ax.set_title("Same best answer, very different learning")
ax.legend()
plt.show()

## 5. What just happened?

- **No baseline, +10:** every sampled answer has a big positive score, so every pick gets pushed up, the wrong ones too. Whichever answer happens to be sampled early snowballs. Many runs lock onto a wrong answer.
- **Mean baseline, +10:** subtracting the batch average cancels the +10 exactly. The line is identical to the run without the bonus. Only *better than usual* gets pushed up.

That is the whole reason GRPO compares answers within a group: the group's average is the baseline.

## 6. Your turn

Try each change, rerun sections 3 and 4, and write one sentence about what you see:

1. Set `lr=2.0` in `train`. Faster? Steadier?
2. Set `batch_size=2`. What happens to the spread?
3. Change `REWARDS` so `no idea` scores 0.6. Does the model still learn `7`?

In [ ]:
# Example: a bigger learning rate
fast = np.stack([train(lr=2.0, seed=s) for s in range(20)])
print(f"lr=2.0: P(7) after 10 steps {fast[:, 10].mean():.2f} (spread {fast[:, 10].std():.2f})")

Next lesson: **Groups and advantages: the heart of GRPO**.